In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,Path,Label
0,train_0,emoji_dataset/train/train_0.png,2
1,train_1,emoji_dataset/train/train_1.png,3
2,train_2,emoji_dataset/train/train_2.png,4
3,train_3,emoji_dataset/train/train_3.png,3
4,train_4,emoji_dataset/train/train_4.png,3


In [40]:
import torch

epochs = 11
batch_size = 32
learning_rate = 0.0008

device = torch.device("cuda")
device

device(type='cuda')

In [41]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

train_transforms = T.Compose([
    T.RandomHorizontalFlip(),
    T.GaussianBlur(kernel_size=3),
    T.ToTensor()
])

valid_transforms = T.Compose([
    T.ToTensor()
])

class MyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.Path.iloc[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.df.Label.iloc[idx], dtype = torch.int64).view(1)
        return img, label

from sklearn.model_selection import train_test_split
df_1, df_2 = train_test_split(df_train, test_size=0.2, random_state=42)

train_dataset = MyDataset(df_1, train_transforms)
train_loader = DataLoader(train_dataset, batch_size=batch_size)

valid_dataset = MyDataset(df_2, valid_transforms)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

train_dataset[0][0][0][0][0]

tensor(1.)

In [42]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, X):
        X = self.net(X)
        return X

model = CNN().to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model

CNN(
  (net): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): LazyLinear(in_features=0, out_features=64, bias=True)
    (11): ReLU()
    (12): Linear(in_features=64, out_features=32, bias=True)
    (13): ReLU()
    (14): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [43]:
def train(model, device, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}")
    print(f"Training Loss: {total_loss / len(train_loader)}")

def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
    print(f'Test Loss: {test_loss / len(test_loader)}')

In [44]:
for epoch in range(1, epochs+1):
    train(model, device, train_loader, criterion, optimizer, epoch)
    test(model, device, valid_loader, criterion)

Epoch 1
Training Loss: 1.4246121080298173
Test Loss: 1.1230036973953248
Epoch 2
Training Loss: 1.1161782914086391
Test Loss: 1.022206425666809
Epoch 3
Training Loss: 1.0381945465740405
Test Loss: 0.9715867400169372
Epoch 4
Training Loss: 0.9294618101496446
Test Loss: 0.9362238109111786
Epoch 5
Training Loss: 0.8770010361545965
Test Loss: 0.9047033727169037
Epoch 6
Training Loss: 0.9017133006924077
Test Loss: 0.9041296958923339
Epoch 7
Training Loss: 0.8636300328530764
Test Loss: 0.9441986620426178
Epoch 8
Training Loss: 0.8172939796196786
Test Loss: 0.9363404750823975
Epoch 9
Training Loss: 0.8637858252776297
Test Loss: 0.9289333760738373
Epoch 10
Training Loss: 0.8004210191337686
Test Loss: 0.9694143414497376
Epoch 11
Training Loss: 0.7318467317443145
Test Loss: 0.9583172023296356


In [45]:
final_transforms = T.Compose([
    T.ToTensor()
])

class FinalDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.Path.iloc[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img

final_dataset = FinalDataset(df_test, final_transforms)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

final_dataset[0][0][0][0]

tensor(1.)

In [46]:
import numpy as np

model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).squeeze(1).cpu().numpy()
        outputs = np.round(outputs)
        outputs = np.clip(outputs, 1, 5)
        preds.extend(outputs)

preds[:10]

[2.0, 3.0, 1.0, 2.0, 1.0, 2.0, 2.0, 1.0, 1.0, 3.0]

In [47]:
df_preds = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "PredictedLabel": preds
})

df_preds

,SampleID,PredictedLabel
0,test_1500,2.0
1,test_1501,3.0
2,test_1502,1.0
3,test_1503,2.0
4,test_1504,1.0
...,...,...
495,test_1995,3.0
496,test_1996,5.0
497,test_1997,3.0
498,test_1998,2.0


In [48]:
df_preds.to_csv("submission.csv", index=False)